# einops-einsum — worked example 3: Outer product of two vectors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-einsum`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import einsum

t.manual_seed(0)
np.random.seed(0)

## Concept

When two operands share **no** index, einsum performs no contraction at all — it forms the full Cartesian product of their axes. Two 1-D vectors with distinct indices `i` and `j` produce the rank-1 outer-product matrix `out[i, j] = a[i] * b[j]`.

## Worked solution

Goal: given vectors `a` of shape `(i,)` and `b` of shape `(j,)`, build `out[i, j] = a[i] * b[j]`.

1. **Use distinct letters.** Because the two axes are independent, we name them differently: `a` is `'i'` and `b` is `'j'`. No letter is shared, so nothing is contracted.
2. **Keep both indices on the output.** The output pattern `-> i j` lists *both* indices, so each survives. An index that appears on input and output is preserved (broadcast-like), so einsum multiplies every `a[i]` against every `b[j]`.
3. **Shape check.** Input shapes `(i,)` and `(j,)` give output shape `(i, j)` — a full matrix. This is the outer product, equivalent to `a[:, None] * b[None, :]` or `torch.outer(a, b)`.

The lesson: *no shared index* means *no reduction*; listing every input index on the output yields an outer product instead of a contraction.

In [ ]:
def outer_product(a: Tensor, b: Tensor) -> Tensor:
    return einsum(a, b, 'i, j -> i j')


t.manual_seed(0)
a = t.randn(3)
b = t.randn(4)
out = outer_product(a, b)
print('shape     =', tuple(out.shape))
print('close?    ', t.allclose(out, t.outer(a, b), atol=1e-5))